# Raw State Calibration: Development Checks

Regenerate raw recordings and rerun connectivity and nested state selection for fixed-law, measurement-change, drift, recurrent, nonrecurring and nonlinear scenarios.
This smoke grid is a software/development check. Multi-state selection is not a validated neural-regime claim; mixed histories are excluded from instantaneous recovery scores.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.
The complete bounded calibration grid is checkpointed. Individual graph fits do not display their own progress bars.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from dataclasses import asdict

from effectome.experiments.state_calibration import (
    run_state_calibration,
    state_calibration_profile,
    write_state_calibration_report,
)
from notebooks._shared import PROJECT_ROOT, make_run, print_notebook_header, print_stage_status

CALIBRATION_CFG = state_calibration_profile("smoke")
OUTPUT_DIRECTORY = PROJECT_ROOT / "outputs" / "state-calibration" / "smoke"
FORCE = False
run = make_run(PROJECT_ROOT / "outputs" / "analysis" / "validation", "smoke", "raw-state-calibration")
print_notebook_header(
    run, title="Raw state calibration", expected_stages=["state_calibration"], force=FORCE,
)
print(CALIBRATION_CFG)


In [ ]:
report = run.execute(
    "state_calibration", lambda: run_state_calibration(CALIBRATION_CFG),
    config=asdict(CALIBRATION_CFG), force=FORCE,
)
manifest_path, scorecard_path = write_state_calibration_report(report, OUTPUT_DIRECTORY)
print(report["summary"])
print("Manifest:", manifest_path)
print("Scorecard:", scorecard_path)
print_stage_status(run)
